In [136]:
import networkx as nx
import pandas as pd

df = pd.read_csv('..\\..\\data\\edges_20k.csv')
df.head()

,Source,Target,weight
0,UCBXNpF6k2n8dsI6nBH8q4sQ,UCM2ERkgV3P1_6MAyxa51rxA,9
1,UCBXNpF6k2n8dsI6nBH8q4sQ,UCpB959t8iPrxQWj7G6n0ctQ,27
2,UCBXNpF6k2n8dsI6nBH8q4sQ,UCtwD0AlYSlAYv7eXu8UxtEg,1
3,UCM2ERkgV3P1_6MAyxa51rxA,UCpB959t8iPrxQWj7G6n0ctQ,76
4,UCM2ERkgV3P1_6MAyxa51rxA,UCtwD0AlYSlAYv7eXu8UxtEg,2


In [137]:
Graphtype = nx.Graph()
G = nx.from_pandas_edgelist(df, source="Source",target="Target",edge_attr='weight', create_using=Graphtype)

In [138]:
communities =nx.community.louvain_communities(G, weight='weight', resolution=1, threshold=1e-07, max_level=None, seed=None)


In [139]:
len(communities)

24

In [140]:
channelsDf = pd.read_csv("..\\..\\data\\df_channels_en.tsv",sep='\t')
channelsDf = channelsDf.set_index("channel")
channelsDf.loc['UC-lHJZR3Gqxm24_Vd_AJ5Yw']

category_cc               Gaming
join_date             2010-04-29
name_cc                PewDiePie
subscribers_cc         101000000
videos_cc                   3956
subscriber_rank_sb           3.0
weights                    2.087
Name: UC-lHJZR3Gqxm24_Vd_AJ5Yw, dtype: object

In [141]:
for community in communities:
    maxSubscribers = 0
    maxChannelName = ''
    for channel in community:
        if channelsDf.loc[channel]['subscribers_cc'] > maxSubscribers:
            maxSubscribers = channelsDf.loc[channel]['subscribers_cc']
            maxChannelName = channelsDf.loc[channel]['name_cc']
    print(f"Community Leader = {maxChannelName}, Community Size = {len(community)}")

Community Leader = 5-Minute Crafts, Community Size = 3571
Community Leader = shane, Community Size = 2106
Community Leader = Justin Bieber, Community Size = 3348
Community Leader = PewDiePie, Community Size = 3857
Community Leader = T-Series, Community Size = 3553
Community Leader = WORLDSTARHIPHOP, Community Size = 2964
Community Leader = WWE, Community Size = 403
Community Leader = Volga Video, Community Size = 409
Community Leader = Tia Mowry's Quick Fi..., Community Size = 2
Community Leader = PrankCity, Community Size = 15
Community Leader = Learn German with Je..., Community Size = 2
Community Leader = NorthSurvival, Community Size = 2
Community Leader = Zombie1, Community Size = 2
Community Leader = Nritya Shakti, Community Size = 2
Community Leader = Music Nepal, Community Size = 115
Community Leader = Tech Ideas, Community Size = 2
Community Leader = Project Life Mastery..., Community Size = 2
Community Leader = Cocomelon - Nursery ..., Community Size = 83
Community Leader = e

In [142]:
nx.set_node_attributes(G, channelsDf.to_dict(orient='index'))


In [143]:
n, m = G.number_of_nodes(), G.number_of_edges()
deg = dict(G.degree())  # unweighted degree
strength = {u: sum(d["weight"] for _, d in G[u].items()) for u in G.nodes()}
nx.set_node_attributes(G, deg, "degree")
nx.set_node_attributes(G, strength, "strength")

In [144]:
comps = sorted(nx.connected_components(G), key=len, reverse=True)
LCC = G.subgraph(comps[0]).copy() if comps else nx.Graph()
n_lcc, m_lcc = LCC.number_of_nodes(), LCC.number_of_edges()
print(f"Graph: {n:,} nodes, {m:,} edges")
print(f"LCC:   {n_lcc:,} nodes, {m_lcc:,} edges  ({(n_lcc/n if n else 0):.1%} of nodes)")

if n_lcc == 0:
    raise SystemExit("Empty LCC. Check inputs or relax filtering.")

Graph: 20,450 nodes, 1,048,575 edges
LCC:   20,426 nodes, 1,048,563 edges  (99.9% of nodes)


In [ ]:
import numpy as np
communities = nx.community.louvain_communities(LCC, weight="weight", resolution=1.0, seed=42)
mod = nx.community.modularity(LCC, communities, weight="weight")
print(f"Communities (Louvain): {len(communities)}  |  Modularity: {mod:.3f}")

# Map node -> community id
node2comm = {}
for cid, com in enumerate(communities):
    for u in com:
        node2comm[u] = cid
nx.set_node_attributes(LCC, node2comm, "community")

# ==== Node metrics on LCC ====
pr = nx.pagerank(LCC, weight="weight")
nx.set_node_attributes(LCC, pr, "pagerank")

# Participation coefficient & within-module z-score
from collections import defaultdict

k_i = {u: LCC.nodes[u]["strength"] for u in LCC.nodes()}
# weighted ties to each community
kic = {u: defaultdict(float) for u in LCC.nodes()}
for u, v, d in LCC.edges(data=True):
    w = d.get("weight", 1.0)
    cu, cv = node2comm[u], node2comm[v]
    kic[u][cv] += w
    kic[v][cu] += w

# P_i = 1 - sum_c (k_i,c / k_i)^2
P = {}
for u in LCC.nodes():
    ki = k_i[u]
    P[u] = 0.0 if ki <= 0 else (1.0 - sum((w/ki)**2 for w in kic[u].values()))

df_comm = pd.DataFrame({
    "node": list(LCC.nodes()),
    "community": [node2comm[u] for u in LCC.nodes()],
    "strength": [k_i[u] for u in LCC.nodes()]
})
stats = df_comm.groupby("community")["strength"].agg(["mean","std"]).rename(columns={"mean":"mu","std":"sigma"})
z = {}
for u in LCC.nodes():
    c = node2comm[u]
    mu, sigma = stats.loc[c, "mu"], stats.loc[c, "sigma"]
    z[u] = 0.0 if (sigma == 0 or np.isnan(sigma)) else (k_i[u] - mu) / sigma

nx.set_node_attributes(LCC, P, "participation")
nx.set_node_attributes(LCC, z, "z_within")

# ==== Dataframes & summaries ====
node_df = pd.DataFrame({
    "channel_id": list(LCC.nodes()),
    "name_cc": [LCC.nodes[u].get("name_cc", "") for u in LCC.nodes()],
    "category_cc": [LCC.nodes[u].get("category_cc", "") for u in LCC.nodes()],
    "community": [node2comm[u] for u in LCC.nodes()],
    "degree": [LCC.nodes[u]["degree"] for u in LCC.nodes()],
    "strength": [LCC.nodes[u]["strength"] for u in LCC.nodes()],
    "pagerank": [pr[u] for u in LCC.nodes()],
    "participation": [P[u] for u in LCC.nodes()],
    "z_within": [z[u] for u in LCC.nodes()],
})

comm_summary = (
    node_df.groupby("community")
           .agg(n_nodes=("channel_id","count"),
                total_strength=("strength","sum"),
                avg_degree=("degree","mean"),
                avg_strength=("strength","mean"),
                avg_participation=("participation","mean"))
           .sort_values("n_nodes", ascending=False)
           .reset_index()
)

# ==== Quick prints ====
print("\nTop communities by size:")
print(comm_summary.head(10).to_string(index=False))

print("\nTop 10 channels by PageRank:")
print(node_df.sort_values("pagerank", ascending=False)
            .head(10)[["channel_id","name_cc","category_cc","community","degree","strength","pagerank"]]
            .to_string(index=False))

# ==== Save CSVs ====
node_df.to_csv("..\\..\\data\\chan_graph_node_metrics.csv", index=False)
comm_summary.to_csv("..\\..\\data\\chan_graph_community_summary.csv", index=False)
print("\nSaved:")
print("..\\..\\data\\chan_graph_node_metrics.csv")
print("..\\..\\data\\chan_graph_community_summary.csv")

Communities (Louvain): 12  |  Modularity: 0.429

Top communities by size:
 community  n_nodes  total_strength  avg_degree  avg_strength  avg_participation
         4     3575         5269759  129.890070   1474.058462           0.494302
         3     3512         1964050   54.737187    559.239749           0.146476
         5     3210         6807363  125.123676   2120.673832           0.407946
        10     3028         2149659   86.658851    709.927015           0.505229
         9     2792         2014468   89.080946    721.514327           0.509037
         1     2195         6301381  162.676993   2870.788610           0.513121
         2      963         1759564  120.340602   1827.169263           0.579839
         0      534          291611   53.713483    546.088015           0.368042
         6      417          302439   46.681055    725.273381           0.222814
         8      142           66435   39.626761    467.852113           0.280816

Top 10 channels by PageRank:
     

In [180]:
# === ENRICHED COMMUNITY PRINT SUMMARY (Network Topology + Categories + Top Channels) ===
import pandas as pd
import numpy as np
import networkx as nx
from collections import Counter
from math import comb, log

assert "LCC" in locals() and "communities" in locals() and "node_df" in locals() and "node2comm" in locals(), \
    "Run the Louvain analysis cell first."

# ---------- Helpers ----------
def gini(x):
    x = np.asarray([v for v in x if np.isfinite(v) and v >= 0], dtype=float)
    if x.size == 0: return np.nan
    if np.allclose(x, 0): return 0.0
    x = np.sort(x)
    n = x.size
    cum = np.cumsum(x)
    return (n + 1 - 2 * np.sum(cum) / cum[-1]) / n

def entropy_from_counts(counts: Counter):
    total = sum(counts.values())
    if total <= 0: return np.nan
    H = 0.0
    for c in counts.values():
        p = c / total
        if p > 0:
            H -= p * log(p + 1e-15)
    return H

# ---------- Prepare base data ----------
node_in_lcc = set(LCC.nodes())
ndf = node_df[node_df["channel_id"].isin(node_in_lcc)].copy()
ndf["community"] = ndf["channel_id"].map(node2comm)
ndf["category_cc"] = ndf["category_cc"].fillna("")

strength = {u: sum(d["weight"] for _, d in LCC[u].items()) for u in LCC.nodes()}
deg = dict(LCC.degree())
pr = nx.pagerank(LCC, weight="weight")

ndf["strength"] = ndf["channel_id"].map(strength)
ndf["degree"] = ndf["channel_id"].map(deg)
ndf["pagerank"] = ndf["channel_id"].map(pr)

# ---------- Compute per-community metrics ----------
rows = []
m_w_total = float(sum(d["weight"] for *_ , d in LCC.edges(data=True)))
comm_sets = {i: set(c) for i, c in enumerate(communities)}

for cid, c_nodes in comm_sets.items():
    sub = LCC.subgraph(c_nodes).copy()
    n_c = sub.number_of_nodes()
    m_c = sub.number_of_edges()
    if n_c == 0:
        continue
    possible = comb(n_c, 2) if n_c >= 2 else 0
    density = (m_c / possible) if possible > 0 else np.nan
    w_internal = float(sum(d["weight"] for *_, d in sub.edges(data=True)))
    strengths = np.array([strength[u] for u in c_nodes])
    volume = strengths.sum()
    cut_w = 0.0
    for u in c_nodes:
        for v, dat in LCC[u].items():
            if v not in c_nodes:
                cut_w += float(dat.get("weight", 1.0))
    vol_S = volume
    vol_not = 2.0 * m_w_total - volume
    denom = min(vol_S, vol_not) if min(vol_S, vol_not) > 0 else np.nan
    conductance = (cut_w / denom) if denom and denom > 0 else np.nan

    cats = ndf.loc[ndf["community"] == cid, "category_cc"]
    cc = Counter(cats.values)
    H = entropy_from_counts(cc)
    top_cat, top_cnt = (cc.most_common(1)[0] if cc else ("", 0))
    top_share = top_cnt / cats.size if cats.size > 0 else np.nan

    g_strength = gini(strengths)
    rows.append(dict(
        community=cid, n_nodes=n_c, n_edges_internal=m_c,
        density=density, w_internal=w_internal, volume=volume,
        conductance=conductance, category_entropy=H,
        top_category=top_cat or "—", top_category_share=top_share,
        gini_strength=g_strength
    ))

summary_df = pd.DataFrame(rows).sort_values("n_nodes", ascending=False).reset_index(drop=True)

# ---------- PRINT RICH REPORT ----------
max_comms_to_show = 10  # print first N communities
for _, row in summary_df.head(max_comms_to_show).iterrows():
    cid = int(row["community"])
    sub_nodes = ndf[ndf["community"] == cid].copy()
    sub_nodes["strength_rank"] = sub_nodes["strength"].rank(ascending=False, method="first")
    top_nodes = sub_nodes.sort_values("strength", ascending=False).head(10)

    print("="*100)
    print(f"🌌 COMMUNITY {cid}")
    print(f"Size: {int(row['n_nodes'])} nodes, {int(row['n_edges_internal'])} internal edges")
    print(f"Density: {row['density']:.4f} | Conductance: {row['conductance']:.4f}")
    print(f"Total internal weight: {row['w_internal']:.1f} | Volume: {row['volume']:.1f}")
    print(f"Category entropy: {row['category_entropy']:.2f} | Top category: {row['top_category']} ({row['top_category_share']:.1%})")
    print(f"Strength inequality (Gini): {row['gini_strength']:.2f}")
    print("─"*100)

    # Category distribution
    cats = Counter(sub_nodes["category_cc"])
    total_c = sum(cats.values())
    print("📊 Category Distribution:")
    for cat, count in cats.most_common(8):
        print(f"   • {cat or '—':25s}  {count:>5d} ({count/total_c:.1%})")
    if len(cats) > 8:
        print(f"   • ... ({len(cats)-8} more categories)")
    print("─"*100)

    # Top 10 nodes (by strength)
    print("🏆 Top 10 Channels:")
    print(top_nodes[["channel_id","name_cc","category_cc","degree","strength","pagerank"]].to_string(index=False))

    # Interactions among the top 10
    top10_ids = set(top_nodes["channel_id"])
    edges_internal = []
    edges_outgoing = []
    for u in top10_ids:
        for v, dat in LCC[u].items():
            w = float(dat.get("weight", 0.0))
            if v in top10_ids and u < v:
                edges_internal.append((u, v, w))
            elif node2comm[v] != cid:
                edges_outgoing.append((u, v, node2comm[v], w))

    print("\n🔗 Internal interactions among top 10:")
    if edges_internal:
        e_df = pd.DataFrame(edges_internal, columns=["src","dst","weight"]).sort_values("weight", ascending=False)
        print(e_df.head(10).to_string(index=False))
    else:
        print("   (no strong internal links among top 10)")

    # Outgoing edges summary by target community
    if edges_outgoing:
        out_df = pd.DataFrame(edges_outgoing, columns=["src","dst","target_comm","weight"])
        out_summary = out_df.groupby("target_comm")["weight"].sum().sort_values(ascending=False)
        print("\n🌍 Outgoing weight from top 10 to other communities:")
        for t, w in out_summary.head(5).items():
            print(f"   → Community {t}: {w:.1f}")
    else:
        print("\n(no outgoing edges from top 10)")

    print("="*100 + "\n")

# ---------- Quick overall table ----------
print("SUMMARY OVERVIEW (first 10 communities by size):")
print(summary_df.head(10)[[
    "community","n_nodes","density","conductance","category_entropy",
    "top_category","top_category_share","gini_strength"
]].to_string(index=False))


🌌 COMMUNITY 4
Size: 3575 nodes, 109708 internal edges
Density: 0.0172 | Conductance: 0.4394
Total internal weight: 1477002.0 | Volume: 5269759.0
Category entropy: 2.07 | Top category: Gaming (33.2%)
Strength inequality (Gini): 0.85
────────────────────────────────────────────────────────────────────────────────────────────────────
📊 Category Distribution:
   • Gaming                      1188 (33.2%)
   • Entertainment                592 (16.6%)
   • Music                        473 (13.2%)
   • Film and Animation           242 (6.8%)
   • Comedy                       233 (6.5%)
   • Science & Technology         225 (6.3%)
   • Education                    208 (5.8%)
   • People & Blogs               139 (3.9%)
   • ... (8 more categories)
────────────────────────────────────────────────────────────────────────────────────────────────────
🏆 Top 10 Channels:
              channel_id                 name_cc     category_cc  degree  strength  pagerank
UC-lHJZR3Gqxm24_Vd_AJ5Yw             

In [181]:
# === Top-100-per-community visualization + legend labeled by top YouTuber ===
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

# -------- Params you can tweak --------
N_PER_COMM              = 5  # pick strongest (by strength) nodes per community
TOP_EDGES_PER_NODE      = 10  # keep strongest k edges incident to each selected node
GLOBAL_EDGE_PERCENTILE  = 98  # also keep edges >= this global percentile by weight
SEED                    = 42
PNG_OUT = "youniverse_top100_per_comm.png"
# --------------------------------------

# Preconditions
assert all(k in globals() for k in ["LCC","communities","node_df","node2comm"]), \
    "Run the Louvain/community analysis first to define LCC, communities, node_df, node2comm."

rng = np.random.default_rng(SEED)

# --- Global metrics on LCC (used for ranking 'top' & legend) ---
strength_LCC = {u: sum(d["weight"] for _, d in LCC[u].items()) for u in LCC.nodes()}
pagerank_LCC = nx.pagerank(LCC, weight="weight")
meta = node_df.set_index("channel_id").to_dict(orient="index")
def name_of(u):
    nm = (meta.get(u, {}) or {}).get("name_cc")
    return nm if (nm and nm.strip()) else u

# --- Select TOP-100 (by strength) per community ---
selected = set()
for cid, nodes_c in enumerate(communities):
    nodes_c = list(nodes_c)
    nodes_c.sort(key=lambda u: strength_LCC.get(u, 0.0), reverse=True)
    selected.update(nodes_c[:N_PER_COMM])

H = LCC.subgraph(selected).copy()
print(f"Subgraph before edge filtering: {H.number_of_nodes():,} nodes, {H.number_of_edges():,} edges")

if H.number_of_nodes() == 0:
    raise SystemExit("No nodes selected — check inputs or lower N_PER_COMM.")

# --- Edge filtering: per-node top-K + global percentile ---
keep_edges = set()

# per-node top-K
for u in H.nodes():
    nbrs = [(u, v, d.get("weight", 0.0)) for v, d in H[u].items()]
    nbrs.sort(key=lambda t: t[2], reverse=True)
    for uu, vv, _ in nbrs[:TOP_EDGES_PER_NODE]:
        a, b = (uu, vv) if uu < vv else (vv, uu)
        keep_edges.add((a, b))

# global percentile
if H.number_of_edges() > 0:
    w_all = np.array([d.get("weight", 0.0) for _,_,d in H.edges(data=True)], dtype=float)
    thr = float(np.percentile(w_all, GLOBAL_EDGE_PERCENTILE))
    for u, v, d in H.edges(data=True):
        if d.get("weight", 0.0) >= thr:
            a, b = (u, v) if u < v else (v, u)
            keep_edges.add((a, b))
else:
    thr = float("nan")

H = H.edge_subgraph(list(keep_edges)).copy()
H.remove_nodes_from(list(nx.isolates(H)))
print(f"After edge filtering: {H.number_of_nodes():,} nodes, {H.number_of_edges():,} edges (global thr≈{thr:.4g})")

if H.number_of_nodes() == 0:
    raise SystemExit("Empty after edge filtering — lower thresholds or raise TOP_EDGES_PER_NODE.")

# --- Ensure attributes on H ---
for n in H.nodes():
    H.nodes[n]["community"] = node2comm[n]
    H.nodes[n]["name_cc"] = meta.get(n, {}).get("name_cc","")
    H.nodes[n]["category_cc"] = meta.get(n, {}).get("category_cc","")
# strength in H uses full-LCC strength to keep sizes consistent with influence
for n in H.nodes():
    H.nodes[n]["strength"] = strength_LCC.get(n, 0.0)

# --- Compute top YouTuber per community (for legend), using full LCC ---
top_per_comm = {}
for cid, nodes_c in enumerate(communities):
    if not nodes_c: 
        continue
    tuples = [(-strength_LCC.get(u,0.0), -pagerank_LCC.get(u,0.0), name_of(u), u) for u in nodes_c]
    tuples.sort()
    _, _, _, top_uid = tuples[0]
    top_per_comm[cid] = name_of(top_uid)
# --- Layout with extreme spreading for central blob ---
print("Computing layout…")
# Step 1: Initial layout with maximum repulsion
pos = nx.spring_layout(H, weight="weight", seed=SEED, 
                       k=6.0,  # extreme repulsion
                       iterations=400,  # extensive settling
                       scale=1.5)  # very large canvas

# Step 2: Multi-pass aggressive spreading
xy = np.array([pos[n] for n in H.nodes()])
nodes_list = list(H.nodes())

# Pass 1: Radial explosion from center
center = xy.mean(0)
distances = np.linalg.norm(xy - center, axis=1)
median_dist = np.median(distances)

for i, n in enumerate(nodes_list):
    if distances[i] < median_dist * 0.85:
        direction = (xy[i] - center)
        if np.linalg.norm(direction) < 1e-6:
            # Node at exact center - push in random direction
            direction = rng.standard_normal(2)
        direction = direction / (np.linalg.norm(direction) + 1e-9)
        
        # Super-exponential push
        centrality = 1 - (distances[i] / (median_dist * 0.85))
        push_strength = 2.5 * median_dist * (centrality ** 1.5)
        xy[i] = center + direction * (distances[i] + push_strength)

# Pass 2: Community-based separation (push different communities apart more)
for _ in range(3):  # multiple iterations
    for i in range(len(xy)):
        comm_i = H.nodes[nodes_list[i]]["community"]
        for j in range(i+1, len(xy)):
            comm_j = H.nodes[nodes_list[j]]["community"]
            diff = xy[i] - xy[j]
            dist = np.linalg.norm(diff)
            
            # Different communities = push harder apart, same community = allow closer
            min_dist = 0.35 if comm_i != comm_j else 0.05
            push_strength = 0.8 if comm_i != comm_j else 0.3
            
            if dist < min_dist:
                push = (diff / (dist + 1e-9)) * (min_dist - dist) * push_strength
                xy[i] += push
                xy[j] -= push

# Pass 3: Final anti-overlap with adaptive threshold
distances_new = np.linalg.norm(xy - xy.mean(0), axis=1)
median_new = np.median(distances_new)
for i in range(len(xy)):
    for j in range(i+1, len(xy)):
        diff = xy[i] - xy[j]
        dist = np.linalg.norm(diff)
        # Adaptive threshold based on distance from center
        dist_from_center = (distances_new[i] + distances_new[j]) / 2
        threshold = 0.08 if dist_from_center < median_new * 0.5 else 0.05
        
        if dist < threshold:
            push = (diff / (dist + 1e-9)) * (threshold - dist) * 0.8
            xy[i] += push
            xy[j] -= push

# Normalize
xy = (xy - xy.mean(0)) / (xy.std(0) + 1e-9)
for i, n in enumerate(nodes_list):
    pos[n] = (xy[i,0], xy[i,1])

# --- Visual encodings ---
comms_shown = sorted({H.nodes[n]["community"] for n in H.nodes()})
cmap = plt.cm.tab20
comm2col = {c: cmap(i / max(1, len(comms_shown)-1)) for i, c in enumerate(comms_shown)}
node_colors = [comm2col[H.nodes[n]["community"]] for n in H.nodes()]

svals = np.array([H.nodes[n]["strength"] for n in H.nodes()], dtype=float)
nsizes = 20 * (np.log10(svals + 1) + 1)  # log-scale sizes by influence
nsizes = np.clip(nsizes, 8, 160)

wvals = np.array([d.get("weight", 0.0) for _,_,d in H.edges(data=True)], dtype=float)
ewidths = np.clip(0.25 + np.log10(wvals + 1.0), 0.35, 2.7)

# --- Draw (edges via LineCollection for speed) ---
fig, ax = plt.subplots(figsize=(14, 10), dpi=180)
ax.set_axis_off()

segs = [(pos[u], pos[v]) for u, v in H.edges()]
lc = LineCollection(segs, linewidths=ewidths, colors="#9AA0A6", alpha=0.10, antialiased=False)
ax.add_collection(lc)

xy = np.array([pos[n] for n in H.nodes()])
ax.scatter(xy[:,0], xy[:,1], s=nsizes, c=node_colors, edgecolors="#111111", linewidths=0.25, alpha=0.95)

# --- Legend: community color → top YouTuber name ---
for cid, color in comm2col.items():
    top_name = top_per_comm.get(cid, f"Community {cid}")
    if len(top_name) > 25:
        top_name = top_name[:24] + "…"
    plt.scatter([], [], c=[color], s=70, label=f"{top_name}  (C{cid})")

plt.legend(loc="lower left", ncol=3, fontsize=6, frameon=False,
           title="Communities labeled by top YouTuber")

plt.title(
    f"YouNiverse — Top {N_PER_COMM}/community • big edges only\n"
    f"Per-node top-{TOP_EDGES_PER_NODE} + ≥P{GLOBAL_EDGE_PERCENTILE:.0f} • Size=log(strength) • Legend by top YouTuber",
    fontsize=11, pad=10
)
plt.tight_layout()
plt.savefig(PNG_OUT, bbox_inches="tight")
plt.close()
print(f"✅ Saved image → {PNG_OUT}")


Subgraph before edge filtering: 60 nodes, 636 edges
After edge filtering: 60 nodes, 354 edges (global thr≈2200)
Computing layout…
✅ Saved image → youniverse_top100_per_comm.png
